# NLP Chatbot — Complete Jupyter Notebook
### Understanding Intent Classification with TF-IDF + Cosine Similarity
---
**What this project does:**  
We build an NLP chatbot that reads your message, understands *what you are asking about* (called **intent**), and replies with the right answer.

**Files in this project:**
| File | Purpose |
|---|---|
| `knowledge_base.py` | All questions and answers (the brain) |
| `intent_classifier.py` | NLP engine that understands user input |
| `chatbot.py` | Main chatbot logic |
| `simple_streamlit_app.py` | Web UI to talk to the chatbot |

**NLP pipeline we will follow:**
```
User Input
    ↓  Lowercase + Remove special chars
Clean Text
    ↓  Remove stopwords + Stem words
Preprocessed Text
    ↓  TF-IDF Vectorizer
Number Vector
    ↓  Cosine Similarity
Best Matching Intent
    ↓  Pick response
Final Answer
```


## Step 1 — Install and Import Libraries
We need these libraries:
- `nltk` — NLP tools (tokenization, stemming, stopwords)
- `sklearn` — TF-IDF vectorizer and cosine similarity
- `numpy` — math operations
- `matplotlib` / `seaborn` — charts and visualizations
- `pandas` — data tables


In [ ]:
# Run this in terminal once:
# pip install nltk scikit-learn numpy matplotlib seaborn pandas

import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("All libraries imported!")


## Step 2 — Download NLTK Data
We need two NLTK resources:
- `stopwords` — common words like "the", "is", "at" that carry no meaning
- `punkt` — sentence tokenizer


In [ ]:
nltk.download('stopwords', quiet=True)
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

print("NLTK data ready!")
print("Stopwords sample:", stopwords.words('english')[:10])


## Step 3 — The Knowledge Base
The knowledge base is the chatbot's brain.  
It has **intents** (topics) and each intent has:
- `patterns` — example questions a user might ask
- `responses` — answers the bot can give

We define a small version here so you can see the structure.  
Your full `knowledge_base.py` has many more intents.


In [ ]:
KNOWLEDGE_BASE = {
    "greeting": {
        "patterns": [
            "hello", "hi", "hey", "greetings",
            "good morning", "good afternoon", "howdy"
        ],
        "responses": [
            "Hello! Welcome to the NLP Chatbot. What would you like to learn today?",
            "Hi there! Ask me about NLP, machine learning, or AI topics!"
        ]
    },
    "tokenization": {
        "patterns": [
            "what is tokenization", "explain tokenization",
            "how does tokenization work", "tokenization in nlp",
            "tell me about tokenization", "types of tokenization"
        ],
        "responses": [
            "Tokenization splits text into smaller units called tokens. "
            "Example: 'The cat sat' becomes ['The', 'cat', 'sat']."
        ]
    },
    "stemming": {
        "patterns": [
            "what is stemming", "explain stemming",
            "how does stemming work", "stemming in nlp",
            "tell me about stemming", "porter stemmer"
        ],
        "responses": [
            "Stemming reduces words to their root form. "
            "Example: 'playing', 'played', 'plays' all become 'play'."
        ]
    },
    "sentiment_analysis": {
        "patterns": [
            "what is sentiment analysis", "explain sentiment analysis",
            "how does sentiment analysis work", "sentiment detection",
            "opinion mining", "emotion detection in text"
        ],
        "responses": [
            "Sentiment analysis detects the emotional tone of text "
            "(positive, negative, neutral). Used in product reviews and social media."
        ]
    },
    "machine_learning": {
        "patterns": [
            "what is machine learning", "explain machine learning",
            "how does machine learning work", "tell me about ml",
            "machine learning definition", "ml algorithms"
        ],
        "responses": [
            "Machine learning is a type of AI where computers learn "
            "from data to make predictions without being explicitly programmed."
        ]
    },
    "goodbye": {
        "patterns": [
            "bye", "goodbye", "see you", "farewell",
            "catch you later", "take care", "quit"
        ],
        "responses": [
            "Goodbye! Thanks for using the NLP Chatbot. See you next time!"
        ]
    }
}

FALLBACK_RESPONSES = [
    "I am not sure I understand. Try asking about tokenization, stemming, or machine learning!",
    "That is outside my knowledge. Ask me about NLP or AI topics!",
    "Hmm, I did not get that. Try rephrasing your question."
]

print("Knowledge base loaded!")
print("Total intents:", len(KNOWLEDGE_BASE))
print("Intents:", list(KNOWLEDGE_BASE.keys()))


## Step 4 — Text Preprocessing
Before we can compare the user's message to our patterns, we need to **clean** the text.

Preprocessing steps:
1. **Lowercase** — "Hello" and "hello" should be the same
2. **Remove special characters** — remove punctuation and numbers
3. **Remove stopwords** — remove "the", "is", "a", "at" etc.
4. **Stem words** — reduce "running", "runs", "ran" → "run"

This makes matching much more accurate.


In [ ]:
stemmer    = PorterStemmer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    # Step 1: lowercase
    text = text.lower()

    # Step 2: remove special characters and digits
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Step 3: split into words
    words = text.split()

    # Step 4: remove stopwords and apply stemming
    words = [
        stemmer.stem(word)
        for word in words
        if word not in stop_words and len(word) > 0
    ]

    return ' '.join(words)

# --- Test it ---
test_sentences = [
    "What is tokenization?",
    "How does stemming work in NLP?",
    "Tell me about machine learning algorithms!",
    "Hello, good morning!"
]

print("Preprocessing examples:")
print("-" * 55)
for s in test_sentences:
    print("Original  :", s)
    print("Processed :", preprocess(s))
    print()


## Step 5 — Visualize Preprocessing Effect
Let us see exactly how preprocessing changes the words.  
We will compare raw words vs processed words for a few sentences.


In [ ]:
examples = [
    "What is tokenization in NLP?",
    "How does stemming work?",
    "Explain machine learning algorithms",
    "Tell me about sentiment analysis"
]

rows = []
for sent in examples:
    raw_words  = sent.lower().split()
    proc_words = preprocess(sent).split()
    rows.append({
        'Original'  : sent,
        'Raw words' : len(raw_words),
        'After preprocessing' : len(proc_words),
        'Words removed' : len(raw_words) - len(proc_words),
        'Processed text' : preprocess(sent)
    })

df_pre = pd.DataFrame(rows)
print(df_pre.to_string(index=False))


In [ ]:
# Bar chart — word count before vs after preprocessing
fig, ax = plt.subplots(figsize=(10, 4))

x      = range(len(examples))
width  = 0.35
raw    = [len(s.lower().split()) for s in examples]
proc   = [len(preprocess(s).split()) for s in examples]
labels = ['Sent 1', 'Sent 2', 'Sent 3', 'Sent 4']

bars1 = ax.bar([i - width/2 for i in x], raw,  width, label='Before', color='#93c5fd', edgecolor='white')
bars2 = ax.bar([i + width/2 for i in x], proc, width, label='After',  color='#6ee7b7', edgecolor='white')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            str(int(bar.get_height())), ha='center', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            str(int(bar.get_height())), ha='center', fontsize=10)

ax.set_xticks(list(x))
ax.set_xticklabels(labels)
ax.set_ylabel('Word count')
ax.set_title('Word count before vs after preprocessing', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('preprocessing_chart.png', dpi=150)
plt.show()
print("Chart saved as preprocessing_chart.png")


## Step 6 — TF-IDF Vectorization
**TF-IDF** = Term Frequency × Inverse Document Frequency

It converts text into numbers so the computer can compare them.

- **TF (Term Frequency):** How often does a word appear in this sentence?
- **IDF (Inverse Document Frequency):** How rare is the word across all sentences?

A word that appears often in ONE sentence but rarely elsewhere gets a HIGH score.  
Common words like "the" appear everywhere so they get a LOW score.

Result: each sentence becomes a **vector of numbers**.


In [ ]:
# Collect all patterns from the knowledge base and preprocess them
all_patterns    = []
pattern_to_intent = []

for intent, data in KNOWLEDGE_BASE.items():
    for pattern in data['patterns']:
        all_patterns.append(preprocess(pattern))
        pattern_to_intent.append(intent)

print("Total patterns:", len(all_patterns))
print()
print("First 8 patterns after preprocessing:")
for i, (p, intent) in enumerate(zip(all_patterns[:8], pattern_to_intent[:8])):
    print("  [" + intent + "]  " + p)


In [ ]:
# Build TF-IDF matrix
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(all_patterns)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Rows = number of patterns:", tfidf_matrix.shape[0])
print("Cols = number of unique words (vocabulary):", tfidf_matrix.shape[1])
print()
print("Vocabulary sample (first 20 words):")
vocab = vectorizer.get_feature_names_out()
print(list(vocab[:20]))


In [ ]:
# Visualize TF-IDF matrix as heatmap (first 12 patterns, first 15 words)
sample_matrix = tfidf_matrix[:12, :15].toarray()
sample_words  = vocab[:15]
sample_labels = [pattern_to_intent[i] + " " + str(i) for i in range(12)]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    sample_matrix,
    xticklabels=sample_words,
    yticklabels=sample_labels,
    cmap='Blues', annot=True, fmt='.2f',
    linewidths=0.5, ax=ax
)
ax.set_title('TF-IDF Matrix — first 12 patterns x first 15 words', fontweight='bold')
ax.set_xlabel('Words (vocabulary)')
ax.set_ylabel('Patterns')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('tfidf_heatmap.png', dpi=150)
plt.show()
print("Heatmap saved as tfidf_heatmap.png")


## Step 7 — Cosine Similarity
Once we have TF-IDF vectors, we compare the **user's input vector** against all **pattern vectors**.

**Cosine Similarity** measures the angle between two vectors:
- Score = **1.0** → perfect match
- Score = **0.5** → partial match
- Score = **0.0** → no match at all

We pick the intent whose pattern has the **highest similarity score**.  
If the best score is below a threshold (0.3), we say we did not understand.


In [ ]:
def classify_intent(user_input, threshold=0.3):
    # Preprocess user input
    processed = preprocess(user_input)

    if not processed:
        return None, 0.0

    # Vectorize user input using the same fitted vectorizer
    user_vector = vectorizer.transform([processed])

    # Calculate cosine similarity with every pattern
    similarities = cosine_similarity(user_vector, tfidf_matrix)[0]

    # Find the highest similarity score
    best_score = float(np.max(similarities))
    best_idx   = int(np.argmax(similarities))

    if best_score < threshold:
        return None, best_score

    # Return the intent that matched
    return pattern_to_intent[best_idx], best_score

# --- Test with some example inputs ---
test_inputs = [
    "what is tokenization",
    "explain how stemming works",
    "tell me about machine learning",
    "hello there",
    "random pizza question xyz"
]

print("Intent Classification Test:")
print("-" * 55)
for inp in test_inputs:
    intent, score = classify_intent(inp)
    result = intent if intent else "NOT RECOGNIZED"
    print("Input  : " + inp)
    print("Intent : " + result + "  |  Score : " + str(round(score * 100, 1)) + "%")
    print()


## Step 8 — Visualize Similarity Scores
Let us see how the similarity scores are distributed for a test input.  
This helps understand how confident the classifier is.


In [ ]:
# Pick one test input and show all similarity scores
test_query  = "how does stemming work"
processed_q = preprocess(test_query)
user_vec    = vectorizer.transform([processed_q])
sims        = cosine_similarity(user_vec, tfidf_matrix)[0]

# Build a dataframe of scores
sim_df = pd.DataFrame({
    'Pattern' : all_patterns,
    'Intent'  : pattern_to_intent,
    'Score'   : [round(float(s), 3) for s in sims]
}).sort_values('Score', ascending=False).head(10)

print('Top 10 matches for: "' + test_query + '"')
print(sim_df.to_string(index=False))


In [ ]:
# Bar chart of top 10 similarity scores
COLORS = {
    'greeting'         : '#93c5fd',
    'tokenization'     : '#fcd34d',
    'stemming'         : '#6ee7b7',
    'sentiment_analysis': '#f9a8d4',
    'machine_learning' : '#c4b5fd',
    'goodbye'          : '#fdba74',
}
DEFAULT_COLOR = '#d1d5db'

top10   = sim_df.head(10)
colors  = [COLORS.get(i, DEFAULT_COLOR) for i in top10['Intent']]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(
    top10['Pattern'],
    top10['Score'],
    color=colors, edgecolor='white', height=0.65
)
for bar, score in zip(bars, top10['Score']):
    ax.text(bar.get_width() + 0.005,
            bar.get_y() + bar.get_height() / 2,
            str(round(score, 3)),
            va='center', fontsize=9)

ax.set_xlim(0, max(top10['Score']) + 0.15)
ax.invert_yaxis()
ax.set_xlabel('Cosine Similarity Score')
ax.set_title('Top 10 pattern matches for: "' + test_query + '"', fontweight='bold')
ax.axvline(x=0.3, color='red', linestyle='--', linewidth=1, label='Threshold 0.3')
ax.legend()

patches = [mpatches.Patch(color=c, label=t) for t, c in COLORS.items()]
ax.legend(handles=patches + [plt.Line2D([0],[0], color='red', linestyle='--', label='Threshold')],
          fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig('similarity_scores.png', dpi=150)
plt.show()
print("Chart saved as similarity_scores.png")


## Step 9 — Knowledge Base Analysis
Let us explore the knowledge base to understand how many patterns each intent has.  
More patterns = better recognition.


In [ ]:
# Count patterns and responses per intent
kb_stats = []
for intent, data in KNOWLEDGE_BASE.items():
    kb_stats.append({
        'Intent'   : intent,
        'Patterns' : len(data['patterns']),
        'Responses': len(data['responses']),
    })

df_kb = pd.DataFrame(kb_stats)
print("Knowledge Base Statistics:")
print(df_kb.to_string(index=False))
print()
print("Total patterns  :", df_kb['Patterns'].sum())
print("Total responses :", df_kb['Responses'].sum())


In [ ]:
# Bar chart — patterns per intent
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: patterns per intent
ax1 = axes[0]
clrs = [COLORS.get(i, DEFAULT_COLOR) for i in df_kb['Intent']]
bars = ax1.bar(df_kb['Intent'], df_kb['Patterns'], color=clrs, edgecolor='white', width=0.6)
for bar in bars:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             str(int(bar.get_height())), ha='center', fontsize=10, fontweight='bold')
ax1.set_ylabel('Number of patterns')
ax1.set_title('Patterns per intent', fontweight='bold')
ax1.set_xticklabels(df_kb['Intent'], rotation=30, ha='right')

# Right: donut chart of pattern distribution
ax2 = axes[1]
ax2.pie(df_kb['Patterns'], labels=df_kb['Intent'], colors=clrs,
        autopct='%1.0f%%', startangle=140, wedgeprops=dict(width=0.55))
ax2.set_title('Pattern distribution across intents', fontweight='bold')

plt.tight_layout()
plt.savefig('kb_analysis.png', dpi=150)
plt.show()
print("Chart saved as kb_analysis.png")


## Step 10 — Build the Full Chatbot
Now we put everything together into a simple chatbot class.  
This is a simplified version of your `chatbot.py` file.


In [ ]:
class NLPChatbot:
    def __init__(self):
        self.history = []

    def get_response(self, user_input):
        intent, confidence = classify_intent(user_input)

        if intent and intent in KNOWLEDGE_BASE:
            response = random.choice(KNOWLEDGE_BASE[intent]['responses'])
        else:
            response = random.choice(FALLBACK_RESPONSES)

        self.history.append({
            'input'     : user_input,
            'intent'    : intent,
            'confidence': round(confidence, 3),
            'response'  : response[:80] + '...' if len(response) > 80 else response
        })
        return response, intent, confidence

bot = NLPChatbot()
print("Chatbot ready!")


In [ ]:
# Test the chatbot with several inputs
test_inputs = [
    "hello",
    "what is tokenization",
    "explain stemming",
    "tell me about machine learning",
    "what is sentiment analysis",
    "bye",
    "random question about pizza"
]

print("Chatbot Test Session")
print("=" * 60)
for user_msg in test_inputs:
    response, intent, conf = bot.get_response(user_msg)
    print("You    : " + user_msg)
    print("Intent : " + str(intent) + "  (" + str(round(conf * 100, 1)) + "%)")
    print("Bot    : " + (response[:90] + "..." if len(response) > 90 else response))
    print("-" * 60)


## Step 11 — Accuracy Evaluation
We test every pattern in the knowledge base against the classifier.  
If the predicted intent matches the actual intent → correct.

This tells us how good the chatbot is at recognising questions.


In [ ]:
correct = 0
total   = 0
results = []

for true_intent, data in KNOWLEDGE_BASE.items():
    for pattern in data['patterns']:
        pred_intent, score = classify_intent(pattern)
        is_correct = pred_intent == true_intent
        if is_correct:
            correct += 1
        total += 1
        results.append({
            'Pattern'         : pattern,
            'True Intent'     : true_intent,
            'Predicted Intent': str(pred_intent),
            'Score'           : round(score, 3),
            'Correct'         : is_correct,
        })

accuracy = round(correct / total * 100, 1)
print("Accuracy  : " + str(accuracy) + "%")
print("Correct   : " + str(correct) + " / " + str(total))
print("Incorrect : " + str(total - correct))


In [ ]:
# Show wrong predictions
df_results = pd.DataFrame(results)
wrong      = df_results[df_results['Correct'] == False]

if len(wrong) == 0:
    print("All patterns classified correctly!")
else:
    print("Wrong predictions:")
    print(wrong[['Pattern','True Intent','Predicted Intent','Score']].to_string(index=False))


In [ ]:
# Accuracy per intent bar chart
intent_acc = {}
for intent in KNOWLEDGE_BASE.keys():
    group   = df_results[df_results['True Intent'] == intent]
    correct = group['Correct'].sum()
    total   = len(group)
    intent_acc[intent] = round(correct / total * 100, 1) if total > 0 else 0

intents = list(intent_acc.keys())
accs    = list(intent_acc.values())
colors  = [COLORS.get(i, DEFAULT_COLOR) for i in intents]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(intents, accs, color=colors, edgecolor='white', width=0.6)

for bar, val in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val) + "%", ha='center', fontsize=10, fontweight='bold')

ax.set_ylim(0, 115)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Classification accuracy per intent', fontweight='bold')
ax.axhline(y=accuracy, color='red', linestyle='--', linewidth=1.2,
           label='Overall: ' + str(accuracy) + "%")
ax.set_xticklabels(intents, rotation=30, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig('accuracy_chart.png', dpi=150)
plt.show()
print("Chart saved as accuracy_chart.png")


## Step 12 — Confidence Score Distribution
Let us look at how confident the classifier is across all patterns.  
High confidence = reliable predictions.  
Low confidence = borderline cases.


In [ ]:
all_scores = df_results['Score'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: histogram of confidence scores
ax1 = axes[0]
ax1.hist(all_scores, bins=15, color='#93c5fd', edgecolor='white', linewidth=0.8)
ax1.axvline(x=0.3,  color='red',    linestyle='--', label='Threshold (0.3)')
ax1.axvline(x=float(np.mean(all_scores)), color='green', linestyle='--',
            label='Mean (' + str(round(float(np.mean(all_scores)), 2)) + ')')
ax1.set_xlabel('Confidence score')
ax1.set_ylabel('Number of patterns')
ax1.set_title('Confidence score distribution', fontweight='bold')
ax1.legend()

# Right: box plot per intent
ax2 = axes[1]
intent_scores = {}
for intent in KNOWLEDGE_BASE.keys():
    grp = df_results[df_results['True Intent'] == intent]['Score'].tolist()
    if grp:
        intent_scores[intent] = grp

box_data   = list(intent_scores.values())
box_labels = list(intent_scores.keys())
bp = ax2.boxplot(box_data, labels=box_labels, patch_artist=True)
for patch, label in zip(bp['boxes'], box_labels):
    patch.set_facecolor(COLORS.get(label, DEFAULT_COLOR))
ax2.set_ylabel('Confidence score')
ax2.set_title('Confidence score per intent', fontweight='bold')
ax2.set_xticklabels(box_labels, rotation=30, ha='right')

plt.tight_layout()
plt.savefig('confidence_dist.png', dpi=150)
plt.show()
print("Chart saved as confidence_dist.png")


## Step 13 — Conversation History Analysis
After running the chatbot, we can look at the full conversation log —  
which intents were triggered, and with what confidence.


In [ ]:
# Show conversation history as a table
if bot.history:
    df_hist = pd.DataFrame(bot.history)
    df_hist['confidence_pct'] = (df_hist['confidence'] * 100).round(1)
    print("Conversation History:")
    print(df_hist[['input','intent','confidence_pct','response']].to_string(index=False))
else:
    print("No history yet. Run Step 10 first.")


In [ ]:
# Pie chart of intents triggered in conversation
if bot.history:
    intent_counts = {}
    for h in bot.history:
        key = h['intent'] if h['intent'] else 'NOT RECOGNIZED'
        intent_counts[key] = intent_counts.get(key, 0) + 1

    fig, ax = plt.subplots(figsize=(8, 5))
    pie_colors = [COLORS.get(k, DEFAULT_COLOR) for k in intent_counts.keys()]
    ax.pie(intent_counts.values(), labels=intent_counts.keys(),
           colors=pie_colors, autopct='%1.0f%%', startangle=140,
           wedgeprops=dict(width=0.6))
    ax.set_title('Intents triggered in this session', fontweight='bold')
    plt.tight_layout()
    plt.savefig('session_intents.png', dpi=150)
    plt.show()
    print("Chart saved as session_intents.png")
else:
    print("No history yet.")


## Step 14 — Summary

### Full pipeline recap:
```
User Input  →  preprocess()  →  TF-IDF vector
                                     ↓
                            cosine_similarity()
                                     ↓
                            Best matching intent
                                     ↓
                            Pick random response
                                     ↓
                               Final answer
```

### Key functions and what they do:

| Function / Class | File | What it does |
|---|---|---|
| `preprocess(text)` | `intent_classifier.py` | Cleans text: lowercase, remove stopwords, stem |
| `TfidfVectorizer` | sklearn | Converts text to number vectors |
| `cosine_similarity` | sklearn | Compares two vectors by angle |
| `classify_intent(input)` | `intent_classifier.py` | Returns best intent + confidence score |
| `NLPChatbot.get_response()` | `chatbot.py` | Calls classifier, picks response |
| `KNOWLEDGE_BASE` | `knowledge_base.py` | Patterns and responses for each intent |
| `FALLBACK_RESPONSES` | `knowledge_base.py` | Used when no intent matches |

### Files to run:
```
# Terminal — just chatbot logic
python chatbot.py

# Streamlit web app
streamlit run simple_streamlit_app.py
```

### How to improve accuracy:
1. Add more patterns per intent in `knowledge_base.py`
2. Lower the threshold (0.3 → 0.2) to match more inputs
3. Add more intents to cover more topics
4. Use lemmatization instead of stemming for better root words
5. Try bigrams in TF-IDF: `TfidfVectorizer(ngram_range=(1,2))`
